 # (Cryptographic) Hash Functions

 In the Algorithms and Data Structures course: Let $U$ be the key set, $m$ the size of the table. Then $h: U \rightarrow 0..(m-1)$ is the hash function and we store the data with key $k$ in the element $T[h(k)]$ of the table. We assume that $|U| >> m$.
 - advantage: $\mathcal{O}(1)$ running time
 - the $h$ function *compresses*

 *Key collision*: For different keys $k_1,k_2$ we have $h(k_1) = h(k_2)$. Since $|U| >> m$, there will definitely be collisions. We want to avoid this, but it can be resolved e.g. by chaining.

 In cryptography we ask for more: it should not be possible to find collisions in polynomial time.

 **Definition**: $H: \{0,1\}^* \rightarrow \{0,1\}^n$

 Some commonly used hash algorithms:

 | Algorithm | Hash length | Collision | Appearance |
 |------------|-------------|----------|------------|
 | MD5        | 128 bit     | explicit | 1992       |
 | SHA-1      | 160 bit     | explicit | 1995       |
 | SHA-256    | 256 bit     | -        | 2001       |
 | SHA-512    | 512 bit     | -        | 2001       |
 | SHA3-256   | 256 bit     | -        | 2016       |
 | SHA3-512   | 512 bit     | -        | 2016       |
 | Blake2b    | 512 bit     | -        | 2015       |
 | Blake2s    | 256 bit     | -        | 2015       |

 [Python hashlib](https://docs.python.org/3/library/hashlib.html)

In [ ]:
import hashlib

hashlib.algorithms_guaranteed


In [ ]:
hashlib.algorithms_available


In [ ]:
hashlib.sha256(b'My secret Password').digest()


In [ ]:
hashlib.sha256(b'My secret Password').hexdigest()


 ## Collision search using the birthday paradox

 **Question**: What is the probability that two people out of $q$ people have the same birthday? (We assume that someone is born with equal probability on any day and there is no leap year)

 Analogous to this: Let $y_i = H(x_i)$ be hashes. What is the probability that $y_i = y_j$ for $i \ne j$?

 **Solution**: Let $p(q)$ be the probability that two people out of $q$ people have the same birthday. Then $1 - p(q)$ is the probability that everyone has a different birthday.

 The probability that $q$ people all have different birthdays is: $365 \cdot 364 \cdot \dots \cdot (365-n+1)$. The total number of possible birthdays is: $365^q$, hence $$p(q) = 1 - \frac{365 \cdot 364 \cdot \dots \cdot (365 - q + 1)}{365^q} = 1 - \frac{365!}{365^q(365-q)!}$$ 

This probability can be approximated by $$p(q) \approx 1 - e^{-\frac{q(q-1)}{2N}}$$ for large $q$. This is the probability that two people have the same birthday in a group of $q$ people, where $N = 365$, the number of days in a year.

To get how many people are needed to have a given probability, we can use the inverse of the approximation:
$$\frac{1}{2}(1+\sqrt{1-8N\ln(1-x)})$$

We can see here that the dominant factor is $\sqrt{N}$, which is the square root of the number of possible birthdays, which is significantly less than one could expect.

To further analyze this, we could compute $p(q)$ for various values of $q$ and plot the results to visualize the likelihood of shared birthdays. 

[Desmos graph](https://www.desmos.com/calculator/3udgwjsklj)

 Is such an attack practical?
 It's solvable, but difficult: to find the specific collision, the computed hashes need to be stored, because we don't know in advance which $x, x'$ pairs cause a collision. $2^{64}$ bits is approximately 2.3 million terabytes. Can this be improved?

 ### Small-space birthday attack

 An attack similar to the previous one with *constant* memory requirement:

 Input: $H : \{0,1\}^* \rightarrow \{0, 1\}^n$

 Output: Different $x,x'$ such that $H(x) = H(x')$

  \begin{array}{l}
      x_0 \leftarrow \{0, 1\}^{n+1} \\
      x' := x := x_0 \\
      \mathbf{while}\;\mathbf{true}: \\
      \quad x := H(x) \\
      \quad x' := H(H(x')) \\
      \quad \mathbf{if}\;x=x'\quad \mathbf{break} \\
      x' := x, x := x_0 \\
      \mathbf{while}\;\mathbf{true}: \\
      \quad \mathbf{if}\;H(x) = H(x')\;\mathbf{return}\;x,x'\;\text{and}\;\mathbf{halt} \\
      \quad \mathbf{else}\;x:=H(x), x':=H(x') \\
  \end{array}

In [ ]:
import secrets
from typing import Callable

def smallspace_bd(h: Callable[..., "hashlib._Hash"], upto: int) -> tuple[bytes, bytes]:
    """
    Find a collision in the first `upto` bytes of the output of `h` hash function.
    """

    x0 = secrets.randbits(h().block_size)
    x0 = x0.to_bytes(h().digest_size)
    xp = x = x0
    while True:
        x = h(x).digest()
        xp = h(h(xp).digest()).digest()
        if x[:upto] == xp[:upto]:
            break
    xp = x
    x = x0
    while True:
        if h(x).digest()[:upto] == h(xp).digest()[:upto]:
            return x, xp
        x = h(x).digest()
        xp = h(xp).digest()


In [ ]:
smallspace_bd(hashlib.md5, 2)


In [ ]:
hashlib.md5(b'N\x1c\xe1\xff\x08\xc5\x92C5\xc6\xf3\xb6\x07G\xd2\xc2').hexdigest()


In [ ]:
hashlib.md5(b'\xb8\xad\xa7\x90\xa6\xf9\xc7:%\xa6\xee\x92?WR\x8c').hexdigest()


 ## Hash functions from block ciphers

 **Davies-Meyer construction**: Let $F$ be a block cipher with $n$-bit keys and $\ell$-bit blocks. Let $$h: \{0,1\}^{n + \ell} \rightarrow \{0,1\}^\ell$$ be defined as $$h_i(m_i, h_{i-1}) = F_{m_i}(h_{i-1}) \oplus h_{i-1}$$

 $h_0$ is some fixed initial value.

In [ ]:
from des import *
from utils import *
from padding import *


In [ ]:
def davies_meyer(msg: bytes) -> bytes:
    hi = bytes.fromhex('0123456789abcdef')
    for i in range(0, len(msg), 8):
        des = DES_CBC((msg[i:i+8]+b"\0x00"*7)[:8], iv=bytes(8))
        ct = des.encrypt(hi, False)[8:]
        hi = xor_strings(ct, hi)
    return hi


In [ ]:
davies_meyer(b'cryptography0123').hex()


In [ ]:
des_weak_keys = ['0101010101010101', 
                 '0000000000000000',
                 'fefefefefefefefe',
                 'ffffffffffffffff']
for w in des_weak_keys:
    print(w, davies_meyer(bytes.fromhex(w)))


 **Task**: Produce a collision for the above Davies-Meyer where the message is at least 16 bytes long.

In [ ]:
davies_meyer(b"\x00" * 16)


In [ ]:
davies_meyer(b"\x01" * 16)


 ## Password cracking

 **Task**: Given the following files containing usernames and passwords. The passwords are hashed using some (unknown) algorithm. Try to crack the passwords!
 - `hashed_pwd_1.txt`
 - `hashed_pwd_2.txt`

 A `pwd.txt` file containing 200 passwords is also given. We can use this for cracking.

 [200 most common passwords](https://nordpass.com/most-common-passwords-list/)

In [ ]:
with open('pwd.txt', 'r') as f:
    pwd = [i.strip() for i in f.readlines()]
print(pwd)


In [ ]:
!head hashed_pwd_1.txt


 ### Dictionary attack

 Objective: reduce runtime (cracking time) using a pre-computed data set (which may take a long time to compute)

 [1,493,677,782 passwords](https://crackstation.net/crackstation-wordlist-password-cracking-dictionary.htm), 15 GB

In [ ]:
pwd_dict = sorted([(p, hashlib.sha256(p.encode()).hexdigest()) for p in pwd], key=lambda x: x[1])
for p, h in pwd_dict:
    print(f'{p:<12s} {h}')


In [ ]:
with open('hashed_pwd_1.txt', 'r') as f:
    for line in f:
        uname, pwd = line.strip().split()
        i = [p[1] for p in pwd_dict].index(pwd)
        print(f'{uname} -> {pwd_dict[i][0]}')


 ### Hash chain, Rainbow table

 [Hellman](https://ee.stanford.edu/~hellman/publications/36.pdf) (1980) and [Oechslin](https://lasec.epfl.ch/pub/lasec/doc/Oech03.pdf) (2003)

 Given a $H: \{0, 1\}^* \rightarrow \{0,1\}^n$ hash function and a finite set $P$ of passwords.

 Goal: Create a data set such that for a given $h$ hash we either
 - find a $p \in P$ password such that $h = H(p)$, or
 - determine that there is no $p \in P$ such that $h = H(p)$ holds.

 Idea: define a *reduction* $R: \{0,1\}^n \rightarrow \{0,1\}^*$ function such that $R(h) = p$ (i.e., what turns a hash into a $P$-value). **Caution: This is not the inverse of the hash!**

 For example:

 $\text{abcdef} \rightarrow_\text{H} \text{18cab0cc42} \rightarrow_\text{R} \text{bla4b1} \rightarrow_\text{H} \text{a7f2bba089} \rightarrow_\text{R} \text{keb5ca}$

 From such a sequence, we only store the first and last elements (start and end points). How do we find the password for a given $h$ hash value?
 1. Apply the $R$ and $H$ functions alternately until the value returned by $R$ matches an endpoint.
 2. Starting from the start point corresponding to the given endpoint, compute the chain until we get the password that resulted in the hash.
     - This is not necessarily unique: another password may result in the same hash.

In [ ]:
import random
from string import ascii_lowercase, digits

from tqdm.notebook import tqdm

abc = ascii_lowercase + digits

def reduction(h, k=5):
    return h[-k:].encode()


In [ ]:
h = hashlib.md5(b'abcdef').hexdigest()
h, reduction(h)


In [ ]:
def create_hash_chain(h, red, n_pwd, chain_length):
    hash_chain = {}
    pbar = tqdm(total=int(n_pwd), desc="Create chain")
    while len(hash_chain) < n_pwd:
        random_pwd = ''.join(random.choices(abc, k=5)).encode()
        new_hash = h(random_pwd).hexdigest()
        
        for _ in range(chain_length - 1):
            red_new_hash = red(new_hash)
            new_hash = h(red_new_hash).hexdigest()
        
        hash_chain[random_pwd] = red_new_hash
        pbar.update(int(1))
    pbar.close()
    return hash_chain


In [ ]:
chain = create_hash_chain(hashlib.md5, reduction, 10, 10)
chain


In [ ]:
def crack_passwd(chain, h, red, uhash, chain_length):
    chain = {v: k for k, v in chain.items()}
    hashed_pwd = uhash
    target_pwd = None
    for _ in range(1_000_000):
        # if we can't find an element that is an endpoint
        # after 1_000_000 iterations, we exit the function
        red_hashed_pwd = red(hashed_pwd)
        if red_hashed_pwd in chain:
            target_pwd = chain[red_hashed_pwd]
            break
        else:
            hashed_pwd = h(red_hashed_pwd).hexdigest()
    
    if target_pwd is None:
        return 'Could not find start of chain'
    
    print(f'Target pwd: {target_pwd}')
    for _ in range(100):
        # we try to extend the chain 100 times,
        # in case we find the key
        for _ in range(chain_length):
            hashed_pwd = h(target_pwd).hexdigest()
            if hashed_pwd == uhash:
                return target_pwd
            target_pwd = red(hashed_pwd)
    return 'Could not find password'


In [ ]:
chain = create_hash_chain(hashlib.md5, reduction, 1_000_000, 100)


In [ ]:
random_pwd = ''.join(random.choices(abc, k=5))
print(random_pwd, hashlib.md5(random_pwd.encode()).hexdigest())
random_pwd, crack_passwd(chain, hashlib.md5, reduction, hashlib.md5(random_pwd.encode()).hexdigest(), 100)


 Problems:
 1. There can also be collisions within the chain, i.e., one chain merges into another chain somewhere. This is very difficult to detect.
 2. The reduction function needs to be chosen well

 The collision problem is improved by the *Rainbow table*: we choose as many reduction functions as the length of the chain we want

 ## Salt, Pepper

 1. Salt: random string, appended to the password before hashing. Stored in the database.
     - $h(p + s)$, $h(s + p)$, $h(h(p) + s)$
 2. Pepper: secret string, typically stored in an HSM.

 Real World Example: [How Dropbox securely stores your passwords (2016)](https://dropbox.tech/security/how-dropbox-securely-stores-your-passwords)

In [ ]:
!head salted_hashed_pwd_2.txt


 Further interesting topics:
 - Explicit collision in SHA1 algorithm: https://shattered.io/
     - 9,223,372,036,854,775,808 SHA-1 compressions
     - 110 GPUs, 1 year of computation
     - Brute force (birthday) same as 12,000,000 GPUs, 1 year of computation
     - Deprecated by NIST in 2011
     - Firefox removed it on February 27, 2017
 - For finding MD5 collisions
     - [An attack](https://iacr.org/archive/eurocrypt2005/34940019/34940019.pdf)
     - The best attack finds a collision from $2^{16}$ hash computations (30 seconds on a phone)
 - SHA-1, SHA-224, SHA-256, SHA-384, SHA-512, SHA-512/224, SHA-512/256 [standard](https://csrc.nist.gov/publications/detail/fips/180/4/final)
 - SHA3-224, SHA3-256, SHA3-384, SHA3-512 [standard](https://csrc.nist.gov/publications/detail/fips/202/final)
     - And the [structures](https://csrc.nist.gov/publications/detail/sp/800-185/final) derived from these

 ## Length extension attack

 **Task**: We want to use a hash algorithm for generating MAC tags. We know the following:
 1. The MAC tag was generated as $H(\text{secret} + \text{message})$
 2. We know the `message` and the length of `secret`, but not `secret` itself
 3. We know the MAC tag
 4. The hash algorithm was SHA256

 Show that with the above information, we can modify the message such that we can generate a valid MAC tag for it!

 **Note**: The above attack works for any hash function based on the Merkle-Damgard construction

In [ ]:
from mysha256 import SHA256

password = b'password'
orig_message = b'count=2&food=pizza&time=now&type=songoku'
orig_hash = hashlib.sha256(password + orig_message).hexdigest()
extension = b'&type=ananas'

mysha256 = SHA256(orig_message, extension, orig_hash, len(password))

print(f'Original message: {orig_message}')
print(f'Original hash:    {orig_hash}')
print(f'Extended message: {mysha256.new_message}')
print(f'New hash:         {mysha256.new_hash.hex()}')
print('-' * 85)
print(f'hashlib hash:     {hashlib.sha256(password + mysha256.new_message).hexdigest()}')

